### Fitting Discrete Markov Models

This notebook demonstrates fitting discrete-state continuous-time Markov
models (CTMC) and inferring ancestral states with `infer_ancestral_states_discrete_ctmc()`.
This example fits a discrete-state continuous-time Markov model (Mk) on a phylogeny, estimates transition rates by maximum likelihood, and then computes marginal ancestral-state posteriors conditional on the fitted parameters. For more details on the models [link]

### Fit discrete traits with an Mk model (ML)

This example fits a discrete-state continuous-time Markov model (Mk) on a phylogeny, estimates transition rates by maximum likelihood, and then computes marginal ancestral-state posteriors conditional on the fitted parameters. For more details on the models [link]

In [1]:
import toytree

In [2]:
tree = toytree.rtree.bdtree(40, seed=2)

#### Simulate tip data and fit a model


In [14]:
X = tree.pcm.simulate_discrete_trait(nstates=3, model="ER", name="X", tips_only=True, inplace=True)

In [16]:
tree.draw(node_sizes=10, node_colors=("X", "Set2"), node_mask=False, layout="d");

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="762.3999999999993px" height="338.84000000000003px" viewBox="0 0 762.3999999999993 338.84000000000003" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t05904e7384c246228ca78d86e42ddc87"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11 r12 r13 r14 r15 r16 r17 r18 r19 r20 r21 r22 r23 r24 r25 r26 r27 r28 r29 r30 r31 r32 r33 r34 r35 r36 r37 r38 r39

In [17]:
tree.pcm.fit_discrete_ctmc("X", nstates=3, model="ER")

PCMDiscreteCTMCFitResult(
  model=ER, nstates=3, nparams=1
  log_likelihood=-25.8677
  state_frequencies=[0.3333, 0.3333, 0.3333]
  relative_rates=<array 3x3>
  qmatrix=<array 3x3>
  fixed_rates=<array 3x3>
  fixed_state_frequencies=None
)

In [24]:
res = tree.pcm.infer_ancestral_states_discrete_ctmc("X", nstates=3, model="ER")

In [27]:
res['model_fit']

PCMDiscreteCTMCFitResult(
  model=ER, nstates=3, nparams=1
  log_likelihood=-25.8677
  state_frequencies=[0.3333, 0.3333, 0.3333]
  relative_rates=<array 3x3>
  qmatrix=<array 3x3>
  fixed_rates=<array 3x3>
  fixed_state_frequencies=None
)

In [29]:
res['data']

,X_anc,X_anc_posterior
0,C,"(0.0, 0.0, 1.0)"
1,C,"(0.0, 0.0, 1.0)"
2,C,"(0.0, 0.0, 1.0)"
3,C,"(0.0, 0.0, 1.0)"
4,C,"(0.0, 0.0, 1.0)"
...,...,...
74,A,"(0.9987903043717012, 0.0005168629302067516, 0...."
75,A,"(0.9919687910259721, 0.0020772724841165765, 0...."
76,A,"(0.9823295810977494, 0.004059121896924204, 0.0..."
77,C,"(0.4360279450137914, 0.0709727150027658, 0.492..."


In [9]:
fit = tree.pcm.infer_ancestral_states_discrete_ctmc(
    data="X",
    nstates=3,
    model="ER",
    inplace=False,
)
fit["model_fit"]

PCMDiscreteCTMCFitResult(
  model=ER, nstates=3, nparams=1
  log_likelihood=-50.6323
  state_frequencies=[0.3333, 0.3333, 0.3333]
  relative_rates=<array 3x3>
  qmatrix=<array 3x3>
  fixed_rates=<array 3x3>
  fixed_state_frequencies=None
)

#### Inspect inferred states and probabilities


In [9]:
df = fit["data"]
df.head()

NameError: name 'fit' is not defined

#### Plot inferred states on the tree


In [ ]:
cm = toytree.color.CMAPS["Set2"] if hasattr(toytree, "color") else None
states = df["state_anc"]
# use a simple palette for three states
palette = ["#4c78a8", "#f58518", "#54a24b"]
node_colors = [
    palette[int(states[i])] if str(states[i]) != "nan" else "#c7c7c7"
    for i in range(tree.nnodes)
]
c, a, m = tree.draw(
    layout="r", node_colors=node_colors, node_sizes=14, tip_labels=True, scale_bar=True
)
c

#### Plot posterior probabilities as pie markers

Use `add_node_pie_markers` with the posterior probabilities to
visualize ancestral state uncertainty at each node.


In [ ]:
# infer and store posterior probabilities on the tree
out = tree.pcm.infer_ancestral_states_discrete_ctmc(
    data=traits,
    nstates=3,
    model="ER",
    inplace=True,
)
c, a, m = tree.draw(layout="d", tip_labels=True, scale_bar=True)
tree.annotate.add_node_pie_markers(
    a, "state_anc_posterior", size=10, istroke_width=1, mask=False
)
c

#### Parameter effects

Below are short examples showing how key fitting parameters influence
inference.


In [10]:
import numpy as np

# Example: constrain relative rates (ARD)
rates = np.array([[0, 2.0, 0.5], [1.0, 0, 0.2], [0.8, 1.5, 0]])
fit_rates = tree.pcm.infer_ancestral_states_discrete_ctmc(
    data=traits,
    nstates=3,
    model="ARD",
    fixed_rates=rates,
    inplace=False,
)
fit_rates["model_fit"]

NameError: name 'traits' is not defined

In [ ]:
# Example: constrain state frequencies
freqs = np.array([0.7, 0.2, 0.1])
fit_freqs = tree.pcm.infer_ancestral_states_discrete_ctmc(
    data=traits,
    nstates=3,
    model="ER",
    fixed_state_frequencies=freqs,
    inplace=False,
)
fit_freqs["model_fit"]

In [ ]:
# Example: set a root prior
root_prior = np.array([0.05, 0.05, 0.90])
fit_root = tree.pcm.infer_ancestral_states_discrete_ctmc(
    data=traits,
    nstates=3,
    model="ER",
    root_prior=root_prior,
    inplace=False,
)
fit_root["model_fit"]

In [ ]:
# Example: scale overall rate
fit_slow = tree.pcm.infer_ancestral_states_discrete_ctmc(
    data=traits,
    nstates=3,
    model="ER",
    rate_scalar=0.2,
    inplace=False,
)
fit_fast = tree.pcm.infer_ancestral_states_discrete_ctmc(
    data=traits,
    nstates=3,
    model="ER",
    rate_scalar=2.0,
    inplace=False,
)
fit_slow["model_fit"]
fit_fast["model_fit"]

#### Fossil constraints: with vs without

Compare inference with and without internal-node observations.


In [ ]:
# Without fossil constraint
base = tree.pcm.infer_ancestral_states_discrete_ctmc(
    data=traits,
    nstates=3,
    model="ER",
    inplace=False,
)

# With a fossil constraint on an internal node
fossil_series = traits.copy().reindex(range(tree.nnodes))
fossil_series.loc[tree[-2]._idx] = 1
fossil = tree.pcm.infer_ancestral_states_discrete_ctmc(
    data=fossil_series,
    nstates=3,
    model="ER",
    inplace=False,
)

# Plot a comparison of inferred states
palette = ["#4c78a8", "#f58518", "#54a24b"]


def plot_states(df, title):
    states = df["state_anc"]
    node_colors = [
        palette[int(states[i])] if str(states[i]) != "nan" else "#c7c7c7"
        for i in range(tree.nnodes)
    ]
    c, a, m = tree.draw(
        layout="r",
        node_colors=node_colors,
        node_sizes=12,
        tip_labels=True,
        scale_bar=True,
    )
    a.label.text = title
    return c


c_base = plot_states(base["data"], "no fossil constraint")
c_fossil = plot_states(fossil["data"], "with fossil constraint")
c_base

In [ ]:
c_fossil